intro_to_Language_modeling part1

[link](https://youtu.be/TCH_1BHY58I)

Previous model was for predicting next char when a single char was given and we had only one layer.
Now we will improve the model to handle multiple input char(3 in this case) with multiple hidden layer

Goal: given 3 char we get the next most likely char. This is a classification model - Bcoz, the model is trying to classify the input sequence into one of the 27 possible next characters

In [64]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [65]:
# download the training data
!curl https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt -o names.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  222k  100  222k    0     0   730k      0 --:--:-- --:--:-- --:--:--  732k


In [66]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

build dataset. i.e build x's and y's
-----


In [67]:
# build the vocabulary of characters and lookup mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [68]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:

  print(f'{w=}')
  context = [0] * block_size # create a list with len 3
  for ch in w + '.':
    ix = stoi[ch] # get char's index
    X.append(context) # initially it will be [0,0,0]
    Y.append(ix)
    #print(context, '--->', itos[ix])
    print('given '+''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append. eg [0,0] + [5] = [0,0,5] . where 5 is the index of char 'e'

X = torch.tensor(X) #shape [32, 3]
Y = torch.tensor(Y) #shape [32]

w='emma'
given ... ---> e
given ..e ---> m
given .em ---> m
given emm ---> a
given mma ---> .
w='olivia'
given ... ---> o
given ..o ---> l
given .ol ---> i
given oli ---> v
given liv ---> i
given ivi ---> a
given via ---> .
w='ava'
given ... ---> a
given ..a ---> v
given .av ---> a
given ava ---> .
w='isabella'
given ... ---> i
given ..i ---> s
given .is ---> a
given isa ---> b
given sab ---> e
given abe ---> l
given bel ---> l
given ell ---> a
given lla ---> .
w='sophia'
given ... ---> s
given ..s ---> o
given .so ---> p
given sop ---> h
given oph ---> i
given phi ---> a
given hia ---> .


In [69]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

Build a lookup table for input embeddings.
-----

using this table we could pick the corresponding embedding for the input char


In [70]:
# Here we are using 27X2, as there are 27 char and we going to represent each char with 2 random numbers
C = torch.randn((27, 2)) # this will be used as the input embedding lookup.

In [71]:
# so if we want to lookup for char 'e' whose index is 5 we can get embedding of it as following
C[5]

tensor([0.9054, 0.3895])

Embedding the inputs(i.e X)
-----

In [72]:
# suppose we want to get the embedding for a single item in X  ,say
X[1] # this would be [0, 0, 5] and which in character form would be [..e]

#embedding of X[1]
C[X[1]] # each 3 char of X(..e) is converted to its corresponding embeddings
#C[X[1]].shape # [3, 2]

tensor([[ 2.5864, -0.2090],
        [ 2.5864, -0.2090],
        [ 0.9054,  0.3895]])

In [73]:
# so embedding for the entire input X
emb = C[X]
emb.shape # shape [32, 3, 2].  32 inputs with 3 chars each and each char with two values

torch.Size([32, 3, 2])

Build hidden layer1
-----

In [74]:
#random weights for 6 inputs(6 bcoz, we provide 3 inputs char of 2 values each)
W1 = torch.randn((6, 100)) # here we use 100 neurons just as an example
#random biases for 100 neurons
b1 = torch.randn(100)

In [75]:
# next what we want is to do summazation (emb @ W1)+ b1
# First we convert the 'emb' to shape [32,6], bcoz we cannot matrix multiply 'emb'of shape [32, 3, 2] and weights W1 of shape [6, 100]

In [76]:
# Optional
emb[:,0,:] # all 32 input embeddings with first char out of the 3 chars
emb[:,1,:] # all 32 input embeddings with second char
emb[:,2,:] # all 32 input embeddings with third char

# now we can concat above 3 sequence of shape[32,2] to form a [32,6] shape tensor
torch.cat((emb[:,0,:], emb[:,1,:], emb[:,2,:]), 1).shape # here last arg 1 say to concat along the dim 1

# alternately, we can use torch.unbind instead of maually getting each seperate sequence as above
torch.unbind(emb, 1)
# and then concat
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [77]:
# The efficent way convert [32,3,2] to [32,6] is to use tensor.view.  instead of using the torch.cat and torch.unbind we can use torch.view
# emb.view(32, 6)
# emb.view(-1, 6)# When we give -1,pytorch will infer what shape that -1 will be based on the other shape we supplied

h = torch.tanh(emb.view(-1, 6) @ W1 + b1) #doing summazation and activation-func
h.shape #[32, 100]

torch.Size([32, 100])

Build output layer
-----

In [78]:
#random weights for 100 inputs(100 bcoz, we 100 inputs from previous hidden layer)
W2 = torch.randn((100, 27)) # here we use 27 neurons as we have 27 possible chars
b2 = torch.randn(27)

In [79]:
#logits - refer specifically to the raw, unnormalized output values of a model's final laye
logits = h @ W2 + b2
logits.shape #so, for 32 input we get logits of shape [32, 27]

torch.Size([32, 27])

In [80]:
# output layer activation-fun (softmax activation-fun in this case)
counts = logits.exp() # exponentiated logits to get a fake count
prob = counts / counts.sum(1, keepdim=True) #normalize

#prob[0].sum() # this should be 1

Loss calculation
-----

In [81]:
# now we check what probalities the model has given for the labels(i.e Y)
prob[torch.arange(32), Y] #torch.arange(32) is just mocking inputs index
#log the probablities
prob[torch.arange(32), Y].log() # Since probabilities are always between 0 and 1, taking the logarithm of a probability (unless it's exactly 1) will result in a negative number.
log_likelihood = prob[torch.arange(32), Y].log().mean()
neg_log_likelihood = -log_likelihood

loss = -prob[torch.arange(32), Y].log().mean() # same as above done in a single step
loss

tensor(18.1141)

Refactoring-01
-----

In [82]:
print(f'dataset X:{X.shape} Y:{Y.shape}') # dataset

dataset X:torch.Size([32, 3]) Y:torch.Size([32])


In [83]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

print("total parameters", sum(p.nelement() for p in parameters))

total parameters 3481


In [84]:
# setting grad as required for all parameters
for p in parameters:
  p.requires_grad = True

In [85]:
for i in range(1000):
  # forward pass
  emb = C[X] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)
  counts = logits.exp()
  probs = counts / counts.sum(1, keepdims=True)
  loss = -probs[torch.arange(32), Y].log().mean()
  #print(loss.item())

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  for p in parameters:
    p.data += -0.1 * p.grad

print("loss",loss.item())

loss 0.2561504542827606


Refactoring-02
-----
using pytorch CROSS_ENTROPY to calculate loss

In [86]:
print(f'dataset X:{X.shape} Y:{Y.shape}') # dataset

dataset X:torch.Size([32, 3]) Y:torch.Size([32])


In [87]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

print("total parameters", sum(p.nelement() for p in parameters))

total parameters 3481


In [88]:
# setting grad as required for all parameters
for p in parameters:
  p.requires_grad = True

In [89]:
for i in range(1000):
  # forward pass
  emb = C[X] # (32, 3, 10)
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)
  # counts = logits.exp()
  # probs = counts / counts.sum(1, keepdims=True)
  # loss = -probs[torch.arange(32), Y].log().mean()
  loss = F.cross_entropy(logits, Y) # https://youtu.be/TCH_1BHY58I?t=2005
  #print(loss.item())

  # backward pass
  for p in parameters:
    p.grad = None # setting grads to zero
  loss.backward()

  # update
  for p in parameters:
    p.data += -0.1 * p.grad # 0.1 is the learning rate

print("loss",loss.item())

loss 0.2561509907245636


Refactoring-03
-----
Here we do training with the full dataset

In [90]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words:
  context = [0] * block_size # create a list with len 3
  for ch in w + '.':
    ix = stoi[ch] # get char's index
    X.append(context) # initially it will be [0,0,0]
    Y.append(ix)
    #print('given '+''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append. eg [0,0] + [5] = [0,0,5] . where 5 is the index of char 'e'

X = torch.tensor(X) #shape [228146, 3]
Y = torch.tensor(Y) #shape [[228146]]
print(f'dataset X:{X.shape} Y:{Y.shape}')

dataset X:torch.Size([228146, 3]) Y:torch.Size([228146])


In [91]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

print("total parameters", sum(p.nelement() for p in parameters))

total parameters 3481


In [92]:
# setting grad as required for all parameters
for p in parameters:
  p.requires_grad = True

In [100]:
for i in range(100):
  # forward pass
  emb = C[X] # (228146, 3, 10)
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (228146, 100)
  logits = h @ W2 + b2 # (228146, 27)
  loss = F.cross_entropy(logits, Y) # https://youtu.be/TCH_1BHY58I?t=2005
  #print(loss.item())

  # backward pass
  for p in parameters:
    p.grad = None # setting grads to zero
  loss.backward()

  # update
  for p in parameters:
    p.data += -0.1 * p.grad # 0.1 is the learning rate

print("loss",loss.item())

loss 2.942652940750122


Refactoring-04
-----
Here we do training in mini batches.

By training on mini-batches, you get a more frequent update to the model's parameters, which can lead to faster convergence and can also help the model escape local minima in the loss landscape.


The key benefits of using mini-batches, even with a small batch size like 32, are:

1. Computational Efficiency:
Processing data in small batches is much more computationally efficient than processing the entire dataset at once, especially when the dataset is very large (like your X tensor with over 200,000 data points). It requires less memory and can often be processed in parallel by hardware like GPUs. Training on the full dataset in one go (called batch gradient descent) can be infeasible due to memory constraints.

2. Faster Updates: With mini-batches, the model's parameters are updated after every batch. If you have a large dataset, training on the full dataset would mean waiting until you've processed all data points before making a single update to the parameters. Mini-batch training allows for much more frequent updates, which helps the model converge faster to a good solution.

3. Noise and Regularization: The randomness introduced by selecting different mini-batches in each iteration adds a form of noise to the training process. This noise can actually be beneficial as it helps the model avoid getting stuck in sharp local minima in the loss landscape and can lead to better generalization on unseen data. It acts as a regularizer.

4. Approximation of the Full Gradient: While the gradient calculated on a single mini-batch is a noisy estimate of the true gradient (the gradient calculated on the entire dataset), it's often a good enough approximation to guide the model towards the minimum of the loss function. As the batch size increases, the mini-batch gradient becomes a better approximation of the true gradient, but you lose some of the benefits of faster updates and regularization.

https://youtu.be/TCH_1BHY58I?t=2547

In [102]:
# # build the dataset
# block_size = 3 # context length: how many characters do we take to predict the next one?
# X, Y = [], []
# for w in words:
#   context = [0] * block_size # create a list with len 3
#   for ch in w + '.':
#     ix = stoi[ch] # get char's index
#     X.append(context) # initially it will be [0,0,0]
#     Y.append(ix)
#     #print('given '+''.join(itos[i] for i in context), '--->', itos[ix])
#     context = context[1:] + [ix] # crop and append. eg [0,0] + [5] = [0,0,5] . where 5 is the index of char 'e'

# X = torch.tensor(X) #shape [228146, 3]
# Y = torch.tensor(Y) #shape [[228146]]
# print(f'dataset X:{X.shape} Y:{Y.shape}') # dataset

In [103]:
g = torch.Generator().manual_seed(2147483647) # for reproducibility
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

print("total parameters", sum(p.nelement() for p in parameters))

total parameters 3481


In [104]:
# setting grad as required to all parameters
for p in parameters:
  p.requires_grad = True

In [105]:
# creating random mini-batches for making trainging faster
ix = torch.randint(0, X.shape[0], (32,))


In [106]:
for i in range(100):
  # mini-batch construct - creating 32 random no between 0 and no of inputs(X.shape[0])
  ix = torch.randint(0, X.shape[0], (32,))

  # forward pass
  emb = C[X[ix]] # randomly selected batch. shape (32, 3, 10)
  #print(emb.shape,emb[0])
  h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
  logits = h @ W2 + b2 # (32, 27)

  loss = F.cross_entropy(logits, Y[ix])
  #print(loss.item())

  # backward pass
  for p in parameters:
    p.grad = None # setting grads to zero
  loss.backward()

  # update
  for p in parameters:
    p.data += -0.1 * p.grad # 0.1 is the learning rate

print("loss",loss.item())

loss 3.8658039569854736


In [107]:
# fininding loss on entire dataset after the min-batch training
emb = C[X] # (228146, 3, 2)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (228146, 100)
logits = h @ W2 + b2 # (228146, 27)
loss = F.cross_entropy(logits, Y)
loss

tensor(3.6994, grad_fn=<NllLossBackward0>)